# finalizing model selection

this notebook evaluates the two model families (logistic regression and random forest) with and without smote oversampling, tunes hyper‑parameters using group‑k‑fold cross‑validation, and logs the results to mlflow.


In [13]:
import pathlib, logging
from pathlib import Path
import pandas as pd, numpy as np
from sklearn.model_selection import GroupKFold, GridSearchCV, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (average_precision_score, roc_auc_score, precision_score, recall_score)
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline
import mlflow, mlflow.sklearn

logging.getLogger("mlflow").setLevel(logging.WARNING)

ROOT = Path.cwd()
mlflow.set_tracking_uri(f"sqlite:///{ROOT / 'mlflow.db'}")
mlflow.set_experiment("glof-critical-area")
print("mlflow tracking:", mlflow.get_tracking_uri())


mlflow tracking: sqlite:////Users/kushalthapa/Desktop/Applied ML/Applied_ML_GLOF_prediction/notebooks/mlflow.db


In [14]:
# load data
data = pd.read_csv('glof_clean.csv')
FEATURES = ["mean_depth", "volume_dam", "volume_gap", "uncertainty_frac", "expansion_frac"]
X = data[FEATURES]
y = data["glof_within_50yr"]
group_col = "lake_id" if "lake_id" in data.columns else None
groups = data[group_col] if group_col else np.arange(len(data))
# grouped split to avoid leakage
unique_lakes = data["lake_id"].unique()
train_lakes, test_lakes = train_test_split(list(unique_lakes), test_size=0.2, random_state=42)
train_mask = data["lake_id"].isin(train_lakes)
X_train = X[train_mask]
X_test = X[~train_mask]
y_train = y[train_mask]
y_test = y[~train_mask]
groups_train = groups[train_mask]
groups_test = groups[~train_mask]



TypeError: only integer scalar arrays can be converted to a scalar index

In [ ]:
def log_run(run_name, pipeline, param_grid, sampler_name):
    cv = GroupKFold(n_splits=5)
    search = GridSearchCV(pipeline, param_grid, cv=cv,
                    scoring="average_precision", n_jobs=-1)
    search.fit(X_train, y_train, groups=groups_train)
    best = search.best_estimator_
    proba = best.predict_proba(X_test)[:, 1]
    pred = best.predict(X_test)
    with mlflow.start_run(run_name=run_name):
        mlflow.log_params({k: str(v) for k, v in search.best_params_.items()})
        mlflow.log_param("sampling", sampler_name)
        mlflow.log_param("split", "group by lake")
        mlflow.log_metric("cv_average_precision", search.best_score_)
        mlflow.log_metric("test_average_precision", average_precision_score(y_test, proba))
        mlflow.log_metric("test_roc_auc", roc_auc_score(y_test, proba))
        mlflow.log_metric("test_precision", precision_score(y_test, pred, zero_division=0))
        mlflow.log_metric("test_recall", recall_score(y_test, pred, zero_division=0))
    return {
        "model": run_name,
        "sampler": sampler_name,
        "cv_ap": search.best_score_,
        "test_ap": average_precision_score(y_test, proba),
        "test_auc": roc_auc_score(y_test, proba),
        "precision": precision_score(y_test, pred, zero_division=0),
        "recall": recall_score(y_test, pred, zero_division=0)
    }


In [ ]:
# define pipelines and grids
# logistic regression (no smote)
logreg_pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(max_iter=1000, class_weight="balanced"))
])
logreg_grid = {"model__C": [0.1, 1, 10]}

# logistic + smote
logreg_smote_pipe = ImbPipeline([
    ("scaler", StandardScaler()),
    ("smote", SMOTE(random_state=42, k_neighbors=3)),
    ("model", LogisticRegression(max_iter=1000, class_weight="balanced"))
])
logreg_smote_grid = {"model__C": [0.1, 1, 10]}

# random forest (no smote)
rf_pipe = Pipeline([
    ("model", RandomForestClassifier(n_estimators=100, class_weight="balanced",
                                 random_state=42, n_jobs=-1))
])
rf_grid = {
    "model__n_estimators": [50, 100],
    "model__max_depth": [3, 6, None]
}

# random forest + smote
rf_smote_pipe = ImbPipeline([
    ("smote", SMOTE(random_state=42, k_neighbors=3)),
    ("model", RandomForestClassifier(class_weight="balanced",
                                 random_state=42, n_jobs=-1))
])
rf_smote_grid = {
    "model__n_estimators": [50, 100],
    "model__max_depth": [3, 6, None]
}


In [ ]:
# run searches and collect results
results = []
results.append(log_run("final-logistic", logreg_pipe, logreg_grid, "none"))
results.append(log_run("final-logistic-smote", logreg_smote_pipe, logreg_smote_grid, "SMOTE"))
results.append(log_run("final-rf", rf_pipe, rf_grid, "none"))
results.append(log_run("final-rf-smote", rf_smote_pipe, rf_smote_grid, "SMOTE"))
summary = pd.DataFrame(results).set_index("model")
print(summary.round(3))


                     sampler  cv_ap  test_ap  test_auc  precision  recall
model                                                                    
final-logistic          none  0.835    0.889     0.998      0.437   1.000
final-logistic-smote   SMOTE  0.854    0.899     0.998      0.459   1.000
final-rf                none  0.974    0.998     1.000      1.000   0.956
final-rf-smote         SMOTE  0.972    0.997     1.000      0.977   0.956
